In [ ]:
import json
import os

import pandas as pd
from src import analysis, common

os.environ["CUDA_VISIBLE_DEVICES"] = ""  # Use CPU for evaluation

In [ ]:
# ======================================================================
# Configuration
# ======================================================================
TASK = "steady_flow"
DATASET_ROOT = common.paths.get_dataset_root()
RUN_NAMES = [
    "replace_with_current_run_name_a",
    "replace_with_current_run_name_b",
]

RUN_LABELS = {
    "replace_with_current_run_name_a": "Run A",
    "replace_with_current_run_name_b": "Run B",
}

SHOW_ID_EVALUATION = True
SHOW_OOD_EVALUATION = True

In [ ]:
run_plans = {}
comparison_contracts = {}


def get_run_plan(run_name: str):
    """Resolve a current saved run and its ID/OOD artifact plan."""
    if run_name not in run_plans:
        run_dir = common.paths.resolve_run_output_dir(TASK, run_name)
        run_plans[run_name] = (run_dir, analysis.artifact_service.load_run_artifact_plan(run_dir))
    return run_plans[run_name]


def _artifact_save_root(*, run_dir, dataset_name: str, split: str):
    """Resolve the provenance-bearing artifact root for one comparison split."""
    if split == "eval":
        return common.paths.resolve_id_analysis_dir(run_dir)
    if split == "ood":
        return common.paths.resolve_ood_analysis_dir(run_dir, dataset_name)
    raise ValueError(f"Unsupported comparison split: {split!r}")


def _load_comparison_contract(*, run_dir, dataset_name: str, split: str) -> dict:
    """Load dataset identity and ordered membership used for one artifact cache."""
    save_root = _artifact_save_root(run_dir=run_dir, dataset_name=dataset_name, split=split)
    provenance_path = analysis.artifacts.artifact_provenance_path(save_root)
    with provenance_path.open(encoding="utf-8") as file:
        provenance = json.load(file)

    dataset = provenance.get("dataset")
    selection = provenance.get("selection")
    if not isinstance(dataset, dict) or not isinstance(selection, dict):
        raise TypeError(f"Invalid artifact comparison provenance: {provenance_path}")

    selection_keys = (
        "index_key",
        "full_selected_case_count",
        "effective_case_count",
        "generation_limit",
        "full_ordered_source_indices_sha256",
        "effective_ordered_source_indices_sha256",
    )
    return {
        "provenance_schema_version": provenance.get("provenance_schema_version"),
        "artifact_schema_version": provenance.get("artifact_schema_version"),
        "split_role": provenance.get("split_role"),
        "dataset": dataset,
        "selection": {key: selection.get(key) for key in selection_keys},
    }


def _require_comparable_artifacts(*, run_name: str, run_dir, dataset_name: str, split: str) -> None:
    """Reject cross-run comparisons with different data identity or membership."""
    contract = _load_comparison_contract(
        run_dir=run_dir,
        dataset_name=dataset_name,
        split=split,
    )
    baseline = comparison_contracts.get(split)
    if baseline is None:
        comparison_contracts[split] = (run_name, contract)
        return

    baseline_run_name, baseline_contract = baseline
    if contract != baseline_contract:
        raise RuntimeError(
            f"Artifacts for split {split!r} are not comparable across runs.\n"
            f"Baseline run: {baseline_run_name}\n"
            f"Current run:  {run_name}\n"
            "Dataset identity and ordered saved-split membership must match."
        )


def run_or_load_artifacts_evaluation(*, run_name: str, split: str) -> tuple[str, pd.DataFrame]:
    """Load or generate artifacts and enforce cross-run comparability."""
    run_dir, plan = get_run_plan(run_name)
    dataset_name = plan.id_dataset_name if split == "eval" else plan.ood_dataset_name
    df_raw = analysis.artifact_service.run_or_load_artifacts(
        run_dir=run_dir,
        dataset_name=dataset_name,
        split=split,
        max_cases=None,
        batch_size=1,
        prefer_cuda=False,
        dataset_root=DATASET_ROOT,
        rebuild=False,
    )
    if not df_raw.empty:
        _require_comparable_artifacts(
            run_name=run_name,
            run_dir=run_dir,
            dataset_name=dataset_name,
            split=split,
        )
    return dataset_name, df_raw


def make_run_label(run_name: str) -> str:
    """Map a run directory name to a human-readable label."""
    return RUN_LABELS.get(run_name, run_name)

In [ ]:
# ======================================================================
# Load evaluation data
# ======================================================================
datasets_eval_id = {}
datasets_eval_ood = {}
comparison_contracts.clear()

for run_name in RUN_NAMES:
    print(f"\n=== {run_name} ===")
    label = make_run_label(run_name)

    if SHOW_ID_EVALUATION:
        dataset_name, df_raw_id = run_or_load_artifacts_evaluation(run_name=run_name, split="eval")
        if df_raw_id.empty:
            print(f"[WARN] No ID artifacts available for {run_name} | {dataset_name}")
        else:
            datasets_eval_id[label] = analysis.evaluation.dataframe.build_eval_df(df_raw_id)

    if SHOW_OOD_EVALUATION:
        dataset_name, df_raw_ood = run_or_load_artifacts_evaluation(run_name=run_name, split="ood")
        if df_raw_ood.empty:
            print(f"[WARN] No OOD artifacts available for {run_name} | {dataset_name}")
        else:
            datasets_eval_ood[label] = analysis.evaluation.dataframe.build_eval_df(df_raw_ood)

In [ ]:
if SHOW_ID_EVALUATION:
    panel_id = analysis.evaluation.panel.build_evaluation_panel(
        datasets_eval=datasets_eval_id,
        title="Run Comparison ID",
        sections="all",
    )
    display(panel_id)

In [ ]:
if SHOW_OOD_EVALUATION:
    panel_ood = analysis.evaluation.panel.build_evaluation_panel(
        datasets_eval=datasets_eval_ood,
        title="Run Comparison OOD",
        sections="all",
    )
    display(panel_ood)